# Task 4-2: Transfer Learning (PyTorch)

**What we do**  
Take 3 pretrained ImageNet models — ResNet-50, MobileNetV3-Large, EfficientNet-B0 —  
and fine-tune them on the **TV Channel Detection** dataset.

For each model we try **two variants**:
1. **Frozen** — only train the classification head (everything else frozen)
2. **Fine-tune** — also unfreeze the last convolutional block

This gives us a 3×2 = 6-row results table (model × variant → accuracy).

**Dataset**: `assignment-4/tv-channel-detection-dataset/` with `train/valid/test` splits.  
Class folders differ across splits, so we use the **intersection**.


links:

https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html


https://pytorch.org/tutorials/beginner/finetuning_torchvision_models_tutorial.html

https://pytorch.org/tutorials/beginner/basics/data_tutorial.html

https://www.learnpytorch.io/06_pytorch_transfer_learning/

Chat GPT + Youtube

In [3]:
# Mount Google Drive (Colab only — run this first if your dataset is in Drive)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted at /content/drive")
except Exception:
    print("Not in Colab or drive already mounted — skipping.")

Mounted at /content/drive
Drive mounted at /content/drive


In [4]:
# Cell 1: Imports

import os
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.models import (
    ResNet50_Weights,
    MobileNet_V3_Large_Weights,
    EfficientNet_B0_Weights,
)

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [5]:
# Cell 2: Find dataset

DATASET_CANDIDATES = [
    os.environ.get("DATASET_DIR"),
    os.path.abspath("assignment-4/tv-channel-detection-dataset"),
    os.path.abspath("tv-channel-detection-dataset"),
    "/content/tv-channel-detection-dataset",
    "/content/drive/MyDrive/datasets/tv-channel-detection-dataset",
    "/content/drive/MyDrive/tv-channel-detection-dataset",
]

DATA_DIR = None
for d in DATASET_CANDIDATES:
    if d and os.path.isdir(d):
        DATA_DIR = d
        break

if DATA_DIR is None:
    raise FileNotFoundError("Dataset not found. Set DATASET_DIR or place it under assignment-4/.")

TRAIN_DIR = os.path.join(DATA_DIR, "train")
VALID_DIR = os.path.join(DATA_DIR, "valid")
TEST_DIR  = os.path.join(DATA_DIR, "test")

print("Dataset:", DATA_DIR)
print("Train:", TRAIN_DIR)
print("Valid:", VALID_DIR)
print("Test: ", TEST_DIR)

Dataset: /content/drive/MyDrive/datasets/tv-channel-detection-dataset
Train: /content/drive/MyDrive/datasets/tv-channel-detection-dataset/train
Valid: /content/drive/MyDrive/datasets/tv-channel-detection-dataset/valid
Test:  /content/drive/MyDrive/datasets/tv-channel-detection-dataset/test


In [6]:
# Cell 3: Figure out which classes exist in ALL three splits

def get_subdirs(folder):
    return {d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d))}

CLASS_NAMES = sorted(get_subdirs(TRAIN_DIR) & get_subdirs(VALID_DIR) & get_subdirs(TEST_DIR))
NUM_CLASSES = len(CLASS_NAMES)

print(f"{NUM_CLASSES} classes (intersection across splits)")
print("Examples:", CLASS_NAMES[:5])

18 classes (intersection across splits)
Examples: ['Logo ABC', 'Logo BBC America', 'Logo BBC World News', 'Logo C-SPAN', 'Logo C-SPAN 2']


In [7]:
# Cell 4: Collect image paths + labels

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

def collect_samples(split_dir):
    """Return list of (image_path, class_index) for classes in CLASS_NAMES."""
    samples = []
    for class_idx, class_name in enumerate(CLASS_NAMES):
        folder = os.path.join(split_dir, class_name)
        for fname in os.listdir(folder):
            if fname.lower().endswith(IMG_EXTENSIONS):
                samples.append((os.path.join(folder, fname), class_idx))
    return samples

train_samples = collect_samples(TRAIN_DIR)
valid_samples = collect_samples(VALID_DIR)
test_samples  = collect_samples(TEST_DIR)

print(f"Train: {len(train_samples)},  Valid: {len(valid_samples)},  Test: {len(test_samples)}")

Train: 1083,  Valid: 38,  Test: 29


In [8]:
# Cell 5: Simple Dataset class + image transforms + DataLoaders

class ImageDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        return self.transform(image), label


MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

BATCH_SIZE = 32

train_loader = DataLoader(ImageDataset(train_samples, train_transform), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
valid_loader = DataLoader(ImageDataset(valid_samples, eval_transform),  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(ImageDataset(test_samples,  eval_transform),  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Loaders ready  (batch_size={BATCH_SIZE})")
print(f"  train batches: {len(train_loader)}")
print(f"  valid batches: {len(valid_loader)}")
print(f"  test  batches: {len(test_loader)}")

Loaders ready  (batch_size=32)
  train batches: 34
  valid batches: 2
  test  batches: 1


In [9]:
# Cell 6: Helper — compute accuracy on a dataloader

@torch.no_grad()
def compute_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        outputs = model(images)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
    return 100.0 * correct / max(total, 1)

In [10]:
# Cell 7: Helper — train for N epochs, return val/test accuracy

def train_and_evaluate(model, train_loader, valid_loader, test_loader, lr, epochs):
    # Only optimise parameters that require gradients
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable_params, lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    start = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        num_samples = 0

        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            num_samples += labels.size(0)

        avg_loss = running_loss / max(num_samples, 1)
        print(f"    epoch {epoch}/{epochs}  loss = {avg_loss:.4f}")

    elapsed = time.time() - start
    val_acc  = compute_accuracy(model, valid_loader)
    test_acc = compute_accuracy(model, test_loader)

    print(f"    val_acc = {val_acc:.2f}%   test_acc = {test_acc:.2f}%   time = {elapsed:.1f}s")
    return val_acc, test_acc, elapsed

In [11]:
# Cell 8: Build a pretrained model and replace its classification head

def load_pretrained_model(name):
    """Load a pretrained model and replace the final layer for NUM_CLASSES."""

    if name == "resnet50":
        model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

    elif name == "mobilenet_v3":
        model = models.mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, NUM_CLASSES)

    elif name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, NUM_CLASSES)

    else:
        raise ValueError(f"Unknown model: {name}")

    return model


def get_head(model, name):
    """Return the classification-head module (the part we always train)."""
    if name == "resnet50":
        return model.fc
    return model.classifier  # mobilenet / efficientnet


def get_last_conv_block(model, name):
    """Return the last convolutional block (the part we unfreeze for fine-tuning)."""
    if name == "resnet50":
        return model.layer4
    if name == "mobilenet_v3":
        return model.features[-3:]   # last 3 feature blocks
    return model.features[-2:]       # efficientnet: last 2 feature blocks

In [12]:
# Cell 9: Freeze / unfreeze helpers

def freeze_everything(model):
    """Freeze all parameters."""
    for param in model.parameters():
        param.requires_grad = False


def unfreeze_module(module):
    """Unfreeze all parameters in a module."""
    for param in module.parameters():
        param.requires_grad = True


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [13]:
# Cell 10: Run the experiment — 3 models × 2 variants

MODEL_NAMES = ["resnet50", "mobilenet_v3", "efficientnet_b0"]
EPOCHS = 3
results = []

for name in MODEL_NAMES:
    for variant in ["frozen", "finetune"]:

        print(f"\n{'='*50}")
        print(f"{name}  /  {variant}")
        print(f"{'='*50}")

        # 1. Load pretrained model + new head
        model = load_pretrained_model(name)

        # 2. Freeze everything first
        freeze_everything(model)

        # 3. Always unfreeze the head
        unfreeze_module(get_head(model, name))

        # 4. For fine-tune: also unfreeze the last conv block
        if variant == "finetune":
            unfreeze_module(get_last_conv_block(model, name))

        print(f"  Trainable params: {count_trainable(model):,}")

        # 5. Pick learning rate
        lr = 1e-3 if variant == "frozen" else 1e-4

        # 6. Move to GPU and train
        model = model.to(DEVICE)
        val_acc, test_acc, elapsed = train_and_evaluate(
            model, train_loader, valid_loader, test_loader, lr=lr, epochs=EPOCHS
        )

        results.append({
            "model": name,
            "variant": variant,
            "val_acc": round(val_acc, 2),
            "test_acc": round(test_acc, 2),
            "time_s": round(elapsed, 1),
        })


resnet50  /  frozen
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 210MB/s]


  Trainable params: 36,882
    epoch 1/3  loss = 2.5457
    epoch 2/3  loss = 1.8737
    epoch 3/3  loss = 1.4583
    val_acc = 47.37%   test_acc = 55.17%   time = 979.1s

resnet50  /  finetune
  Trainable params: 15,001,618
    epoch 1/3  loss = 2.6802
    epoch 2/3  loss = 1.7880
    epoch 3/3  loss = 1.0406
    val_acc = 55.26%   test_acc = 51.72%   time = 65.0s

mobilenet_v3  /  frozen
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 206MB/s]


  Trainable params: 1,253,138
    epoch 1/3  loss = 2.0040
    epoch 2/3  loss = 0.9210
    epoch 3/3  loss = 0.5370
    val_acc = 50.00%   test_acc = 55.17%   time = 54.5s

mobilenet_v3  /  finetune
  Trainable params: 3,003,378
    epoch 1/3  loss = 2.6847
    epoch 2/3  loss = 2.0174
    epoch 3/3  loss = 1.3269
    val_acc = 57.89%   test_acc = 55.17%   time = 55.7s

efficientnet_b0  /  frozen
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 235MB/s]


  Trainable params: 23,058
    epoch 1/3  loss = 2.5760
    epoch 2/3  loss = 1.8968
    epoch 3/3  loss = 1.5411
    val_acc = 44.74%   test_acc = 55.17%   time = 57.2s

efficientnet_b0  /  finetune
  Trainable params: 1,152,450
    epoch 1/3  loss = 2.7996
    epoch 2/3  loss = 2.4444
    epoch 3/3  loss = 2.1285
    val_acc = 36.84%   test_acc = 44.83%   time = 57.3s


In [15]:
# Cell 11: Print results table + save CSV

df = pd.DataFrame(results)

print("\nResults (copy to PPT):")
print(df.to_string(index=False))

# Save CSV next to the dataset so it persists on Drive (Colab) or locally
artifacts_dir = os.path.join(DATA_DIR, "..", "artifacts")
os.makedirs(artifacts_dir, exist_ok=True)
csv_path = os.path.join(artifacts_dir, "task4_2_results.csv")
df.to_csv(csv_path, index=False)
print("\nSaved:", os.path.abspath(csv_path))


Results (copy to PPT):
          model  variant  val_acc  test_acc  time_s
       resnet50   frozen    47.37     55.17   979.1
       resnet50 finetune    55.26     51.72    65.0
   mobilenet_v3   frozen    50.00     55.17    54.5
   mobilenet_v3 finetune    57.89     55.17    55.7
efficientnet_b0   frozen    44.74     55.17    57.2
efficientnet_b0 finetune    36.84     44.83    57.3

Saved: /content/drive/MyDrive/datasets/artifacts/task4_2_results.csv
